In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage 

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [32]:
config = {"configurable":{"thread_id":"test-3"}}

questions = [
    "What is 2+2?",
    "What is 5+3?",
    "What is 10-4?",
    "What is 6*2?",
    "What is 15/3?",
    "What is 9+7?"
]


In [33]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(
        content=q
    )]},config)

    print(f"message :{response["messages"][-1].content}")
    print(f"Total message :{len(response["messages"])}")

message :2 + 2 = 4
Total message :2
message :5 + 3 = 8
Total message :4
message :10 - 4 = 6
Total message :6
message :6 * 2 = 12
Total message :8
message :15 / 3 = 5
Total message :10
message :9 + 7 = 16
Total message :6


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city:str)->str:
    """search hotels-returns lang response to use more tokens"""
    return f"""Hotels in {city}: 
            1. grand hotel -5 star , $350/night , spa , pool , gym 
            2. city inn -4 star , $180/night , bussiness center 
            3. budget stay -3 star , $75/night , free wifi  
    """

agent = create_agent(
     model="google_genai:gemini-2.5-flash",
     tools=[search_hotel] ,\
     checkpointer=InMemorySaver(),
     middleware=[SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash",
        trigger=("tokens",550),
        keep=("tokens",200)
     )]

)

In [10]:
config = {"configurable":{"thread_id":"test-1"}}

def count_token(messages):
    total_chars = 0

    for m in messages:
        total_chars += len(str(m.content))

    return total_chars

In [11]:
cities = ["paris", "London", "tokyo", "new york", "dubai", "singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"find hotels in {city}"
                )
            ]
        },
        config=config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(response["messages"])

paris: ~2228 tokens, 12 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='a571d563-539a-49d9-938e-555fdc511f76'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotel', 'arguments': '{"city": "paris"}'}, '__gemini_function_call_thought_signatures__': {'2827ff74-bcb1-4566-adba-8f4b011842d3': 'CrABARFNMg9iheCoq7yb8ZCQxwabF59K8L5LWDHcagY55ICzLHCygL40fKN2yOtsCxWrPt2xEn/DHHcTzQxYa4qL2iQzr8fdM4fZEhSexUFNjye2OTcJRN0n5Lt06GmB3TerbEytkcvtrs7FNdL9CeH9UMubjdSEahvfZ0FlVmDe51xVLrdM5mU51jOEdZ/maVAqc2jR6Z1zcfP+/NyhbYq0Wz+kejf9cOCbHZUoRuLOUG0='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f6244-32d3-7d52-8752-30208c0b966e-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'paris'}, 'id': '2827ff74-bcb1-4566-adba-8f4b011842d3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, '

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 37.354229748s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT